# Mod3D Usage Examples — PyVista rendering

Same tour of the Mod3D bindings as `usage_examples.ipynb`, but rendered with the
**PyVista** backend instead of pythreejs. Each cell builds a shape and renders it
inline as an interactive trame widget.

Requires `pyvista` and the trame stack (`trame`, `trame-vtk`); for the HTML
export shown at the end you may also need `nest_asyncio2`. The same
`PyVistaRenderer` also drives a desktop window (`mode="window"`) and a
self-contained shareable HTML file (`mode="html"`).

In [ ]:
# Shared renderer setup (PyVista backend)
from mod3d.render.pyvista import PyVistaRenderer

def render(*shapes, **kwargs):
    """Quick helper to render one or more shapes with the PyVista backend."""
    r = PyVistaRenderer(window_size=(800, 600))
    for s in shapes:
        r.add_shape(s)
    return r.render(mode="notebook", background="lightgray", **kwargs)

## Geometric Primitives

In [ ]:
from mod3d import gp

# Points, vectors, directions
p = gp.Pnt(1.0, 2.0, 3.0)
v = gp.Vec(1.0, 0.0, 0.0)
d = gp.Dir(0.0, 0.0, 1.0)

# Transformations
trsf = gp.Trsf()
trsf.set_translation(gp.Vec(5.0, 0.0, 0.0))

# Direct transformation methods
ax = gp.Ax1(gp.Pnt(0, 0, 0), gp.Dir(0, 0, 1))
p2 = gp.Pnt(1, 2, 3).translated(gp.Vec(10, 0, 0))
ax2 = ax.rotated(gp.Ax1(gp.Pnt(0, 0, 0), gp.Dir(0, 0, 1)), 1.57)

print(f"Original point: {p}")
print(f"Translated point: {p2}")

## Creating Shapes — Primitives

In [ ]:
from mod3d import BRepBuilderAPI, gp

box = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
render(box)

In [ ]:
sphere = BRepBuilderAPI.MakeSphere(5.0).shape()
render(sphere)

In [ ]:
cylinder = BRepBuilderAPI.MakeCylinder(2.0, 15.0).shape()
render(cylinder)

In [ ]:
cone = BRepBuilderAPI.MakeCone(10.0, 5.0, 20.0).shape()
render(cone)

## Creating Shapes — Edges, Wires, and Faces

In [ ]:
from mod3d import BRepBuilderAPI, gp

p1, p2 = gp.Pnt(0, 0, 0), gp.Pnt(10, 0, 0)
edge = BRepBuilderAPI.MakeEdge(p1, p2).edge()
wire = BRepBuilderAPI.MakeWire(edge).wire()
render(wire)

## B-Spline Curves

In [ ]:
from mod3d import gp, Geom, BRepBuilderAPI

# B-Spline from gp.Pnt list
curve = Geom.BSplineCurve(
    poles=[gp.Pnt(0, 0, 0), gp.Pnt(1, 1, 0), gp.Pnt(2, 1, 0), gp.Pnt(3, 0, 0)],
    knots=[0.0, 1.0, 2.0],
    multiplicities=[3, 1, 3],
    degree=2,
)
edge = BRepBuilderAPI.MakeEdge(curve).edge()
render(edge)

In [ ]:
import numpy as np

# B-Spline from numpy arrays with weights
curve_np = Geom.BSplineCurve(
    poles=np.array([[0, 0, 0], [10, 0, 0], [10, 10, 0], [10, 10, 10]]),
    weights=[1.0, 1.2, 0.8, 0.5],
    knots=[0, 1],
    multiplicities=[4, 4],
    degree=3,
)
edge_np = BRepBuilderAPI.MakeEdge(curve_np).edge()
render(edge_np)

## Boolean Operations

In [ ]:
import mod3d

sphere = mod3d.BRepBuilderAPI.MakeSphere(5.0).shape()
box = mod3d.BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()

# Move box to overlap
trsf = mod3d.gp.Trsf()
trsf.set_translation(mod3d.gp.Vec(-10.0, -10.0, -10.0))
box = box.moved(trsf)

# Fuse (union)
fuse = mod3d.BooleanOp.Fuse(box, sphere)
render(fuse.shape())

In [ ]:
# Common (intersection)
common = mod3d.BooleanOp.Common(sphere, box)
render(common.shape())

In [ ]:
# Cut (subtraction)
cut = mod3d.BooleanOp.Cut(box, sphere)
render(cut.shape())

## Filleting

In [ ]:
from mod3d import BRepFillet, BRepBuilderAPI, BooleanOp, gp

# Create two overlapping shapes and fuse them
s = BRepBuilderAPI.MakeSphere(5.0).shape()
b = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
t = gp.Trsf()
t.set_translation(gp.Vec(-10.0, -10.0, -10.0))
b = b.moved(t)
fuse = BooleanOp.Fuse(b, s)
result = fuse.shape()

# 3D fillet on section edges
fillet_maker = BRepFillet.MakeFillet(result)
for edge in fuse.section_edges():
    fillet_maker.add(1.0, edge)
filleted = fillet_maker.shape()
render(filleted)

## Curve Interpolation

In [ ]:
from mod3d import gp, GeomAPI, BRepBuilderAPI

points = [gp.Pnt(0, 0, 0), gp.Pnt(1, 1, 0), gp.Pnt(2, 0, 0)]
interp = GeomAPI.Interpolate(points)
interp.perform()
curve = interp.curve
edge = BRepBuilderAPI.MakeEdge(curve).edge()
render(edge)

## Distance Computation

In [ ]:
from mod3d import BRepExtrema, BRepBuilderAPI, gp

box1 = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
box2 = BRepBuilderAPI.MakeBox(gp.Pnt(20.0, 0.0, 0.0), 10.0, 10.0, 10.0).shape()

dist = BRepExtrema.DistShapeShape(box1, box2)
print(f"Distance between boxes: {dist.value}")

render(box1, box2)

## Global Properties

In [ ]:
from mod3d import BRepGProp, BRepBuilderAPI

box = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
props = BRepGProp.BRepGProp.linear_properties(box)
print(f"Centre of mass: {props.centre_of_mass}")

render(box)

## Tessellation

In [ ]:
from mod3d import Render, BRepBuilderAPI

box = BRepBuilderAPI.MakeBox(10.0, 20.0, 30.0).shape()
faces, edges = Render.extract_tessellation(box, linear_deflection=0.1)

for triangles, vertices, normals, uvs in faces:
    print(f"{vertices.shape[0]} vertices, {triangles.shape[0]} triangles")

render(box)

## Interactive Rendering — Custom Options

In [ ]:
from mod3d.render.pyvista import PyVistaRenderer
from mod3d import BRepBuilderAPI, gp

# Build a scene with multiple primitives
box = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
sphere = BRepBuilderAPI.MakeSphere(gp.Pnt(15, 5, 5), 4.0).shape()
cyl = BRepBuilderAPI.MakeCylinder(2.0, 12.0).shape()

renderer = PyVistaRenderer(window_size=(800, 600))
renderer.angle_deflection = 5
renderer.linear_deflection = 0.01
renderer.add_shape(box, {'surface_color': '#ff6600'})
renderer.add_shape(sphere, {'surface_color': '#0066ff'})
renderer.add_shape(cyl, {'surface_color': '#00cc44'})
renderer.render(mode="notebook", background='lightgray')

## Sharing — export to a self-contained HTML file

The same scene can be written to a single interactive HTML file (vtk.js), openable
in any browser with no server. This is the PyVista backend's main advantage over the
notebook-only pythreejs renderer.

In [ ]:
renderer.render(mode="html", path="usage_examples_scene.html")